# 03 -- Feature Extraction

**Attribution:** Original FDD implementation and ACC_Z feature-extraction workflow by `Mohsen Rezvani Alile`.
This version migrates the established logic into reusable toolkit helpers while keeping the same
peak-picking, mode-shape summarization, and display structure.

This notebook preserves Mohsen's FDD-focused feature workflow and keeps the feature-extraction outputs that originally appeared in `02_preprocessing.ipynb`:

- Frequency Domain Decomposition (FDD) on prefiltered multichannel `ACC_Z` events
- picked FDD peaks
- signed and absolute mode-shape summaries
- mode-shape plots by structural location
- first-singular-value spectra with picked-peak annotations

Signal conditioning (0.5–20 Hz zero-phase Butterworth band-pass, baseline zeroing, and
`Synchro()` timestamp alignment) is handled by the preprocessing stage; see
`02_preprocessing.ipynb` for the pipeline walkthrough.


In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aquinas_toolkit import AquinasReader
from aquinas_toolkit.feature_extraction import run_acc_z_fdd_workflow

MIN_DURATION_SECONDS = 10.0
DATASET_ROOT = Path("../AQUINAS_DATASET")
SAMPLE_RATE_HZ = 100.0
LOW_HZ = 0.5
HIGH_HZ = 20.0
MAX_EVENTS_PER_SET = 5
TOP_MODE_COUNT = 3
DECKS = ["OLD", "NEW"]
SIDE_LABELS = {"DO": "Downstream side", "UP": "Upstream side"}
MODE_SHAPE_NODE_LAYOUT = [
    {"label": "Left SUP", "span": None, "location": "SUP", "value": 0.0},
    {"label": "S1 MID", "span": "S1", "location": "MID"},
    {"label": "S1 INT", "span": "S1", "location": "INT"},
    {"label": "Pier SUP", "span": None, "location": "SUP", "value": 0.0},
    {"label": "S2 INT", "span": "S2", "location": "INT"},
    {"label": "S2 MID", "span": "S2", "location": "MID"},
    {"label": "Right SUP", "span": None, "location": "SUP", "value": 0.0},
]

fdd_overview = []
fdd_results_by_set = {}

for set_dir in sorted(DATASET_ROOT.glob("AQUINAS_SET*")):
    reader = AquinasReader(set_dir)

    for deck in DECKS:
        try:
            result = run_acc_z_fdd_workflow(
                reader,
                min_duration_seconds=MIN_DURATION_SECONDS,
                deck=deck,
                max_events=MAX_EVENTS_PER_SET,
                sampling_rate_hz=SAMPLE_RATE_HZ,
                low_hz=LOW_HZ,
                high_hz=HIGH_HZ,
                nperseg=1024,
                noverlap=512,
                n_peaks=TOP_MODE_COUNT,
            )
        except ValueError:
            continue

        result_key = f"{reader.set_name} | {deck}"
        fdd_overview.append(
            {
                "Dataset": reader.set_name,
                "Deck": deck,
                "ACC_Z channels": int(result["common_events"]["channel_count"].iloc[0]),
                "Common events available": len(result["common_events"]),
                "Aligned events used": len(result["selected_events"]),
                "Top FDD peak [Hz]": float(result["peak_table"].iloc[0]["frequency_hz"]),
            }
        )
        fdd_results_by_set[result_key] = result

display(
    pd.DataFrame(fdd_overview)
    .style.hide(axis="index")
    .format({
        "ACC_Z channels": "{:,.0f}",
        "Common events available": "{:,.0f}",
        "Aligned events used": "{:,.0f}",
        "Top FDD peak [Hz]": "{:.2f}",
    })
    .set_caption(
        f"Per-deck ACC_Z events after {MIN_DURATION_SECONDS:.0f} s duration filtering, 0.5-20 Hz zero-phase band-pass, and FDD"
    )
)



In [ ]:
for result_key, result in fdd_results_by_set.items():
    display(Markdown(f"### {result_key}"))

    display(
        result["peak_table"][["frequency_hz", "singular_value"]]
        .rename(columns={"frequency_hz": "Frequency [Hz]", "singular_value": "Singular value"})
        .style.hide(axis="index")
        .format({"Frequency [Hz]": "{:.2f}", "Singular value": "{:.4e}"})
        .set_caption(f"{result_key} — picked FDD peaks")
    )

    display(
        result["signed_component_table"]
        .style
        .format("{:+.3f}")
        .set_caption(f"{result_key} — phase-aligned signed mode-shape components")
    )

    display(
        result["amplitude_table"]
        .style
        .format("{:.3f}")
        .set_caption(f"{result_key} — absolute mode-shape amplitudes")
    )

    display(
        result["phase_table"]
        .style
        .format("{:.1f}")
        .set_caption(f"{result_key} — location-based mode-shape phases [deg]")
    )

    for peak_rank, peak in enumerate(result["peak_table"].itertuples(index=False), start=1):
        peak_rows = result["mode_shape_locations"][result["mode_shape_locations"]["peak_rank"] == peak_rank].copy()
        peak_rows = peak_rows.sort_values(["side", "span", "location"])

        figure, axes = plt.subplots(2, 1, figsize=(12, 7), sharey=True)

        for axis, side in zip(axes, ["DO", "UP"], strict=True):
            side_rows = peak_rows[peak_rows["side"] == side].copy()
            plot_rows = []

            for node in MODE_SHAPE_NODE_LAYOUT:
                if node["span"] is None:
                    plot_rows.append(
                        {
                            "label": node["label"],
                            "signed_component": float(node["value"]),
                            "color": "tab:gray",
                            "is_support": True,
                        }
                    )
                    continue

                matching_rows = side_rows[
                    (side_rows["span"] == node["span"])
                    & (side_rows["location"] == node["location"])
                ]
                if matching_rows.empty:
                    plot_rows.append(
                        {
                            "label": node["label"],
                            "signed_component": np.nan,
                            "color": "tab:gray",
                            "is_support": False,
                        }
                    )
                    continue

                signed_component = float(matching_rows.iloc[0]["mode_shape_signed_component"])
                plot_rows.append(
                    {
                        "label": node["label"],
                        "signed_component": signed_component,
                        "color": "tab:blue" if signed_component >= 0 else "tab:orange",
                        "is_support": False,
                    }
                )

            plot_frame = pd.DataFrame(plot_rows)
            x_values = np.arange(len(plot_frame))

            axis.bar(
                x_values,
                plot_frame["signed_component"].fillna(0.0),
                color=plot_frame["color"],
                alpha=0.45,
                width=0.72,
            )
            axis.plot(
                x_values,
                plot_frame["signed_component"],
                color="black",
                linewidth=1.5,
                marker="o",
                markersize=5,
            )

            support_mask = plot_frame["is_support"].to_numpy()
            axis.scatter(
                x_values[support_mask],
                plot_frame.loc[support_mask, "signed_component"],
                color="black",
                marker="s",
                s=36,
                zorder=3,
            )

            axis.axhline(0.0, color="black", linewidth=0.9)
            axis.set_ylim(-1.05, 1.05)
            axis.set_ylabel("Signed component")
            axis.set_xticks(x_values, plot_frame["label"], rotation=35, ha="right")
            axis.set_title(SIDE_LABELS[side])
            axis.grid(axis="y", alpha=0.3)

        axes[-1].set_xlabel("Location along deck")
        figure.suptitle(
            f"{result_key} — mode shape for peak {peak_rank} at {peak.frequency_hz:.2f} Hz"
        )
        figure.tight_layout()
        plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(result["frequencies_hz"], result["first_singular_value"], linewidth=1.5)
    for peak in result["peak_table"].itertuples(index=False):
        plt.axvline(peak.frequency_hz, color="tab:red", linestyle="--", linewidth=0.9, alpha=0.7)
        plt.text(
            peak.frequency_hz,
            peak.singular_value,
            f"{peak.frequency_hz:.2f} Hz",
            rotation=90,
            va="bottom",
            ha="right",
            fontsize=8,
        )
    plt.xlim(LOW_HZ, HIGH_HZ)
    plt.xlabel("Frequency [Hz]")
    plt.ylabel("First singular value")
    plt.title(f"{result_key} — FDD spectrum with picked peaks")
    plt.grid(alpha=0.3)
    plt.show()
